## Step 5: Split Integrity, Metrics Toolkit and Baseline

In [1]:
import pandas as pd, numpy as np, sys, hashlib
import matplotlib.pyplot as plt, seaborn as sns
sys.path.append("../src")

train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")
ft_train = pd.read_csv("../data/features_train.csv")
ft_test  = pd.read_csv("../data/features_test.csv")

target = "DEFAULT"
X_train, y_train = ft_train.drop(columns=target), ft_train[target]
X_test,  y_test  = ft_test.drop(columns=target),  ft_test[target]

In [2]:
#Split integrity checks
#Sizes and default rates
print("Train:", X_train.shape, "default rate:", round(y_train.mean(), 4))
print("Test :", X_test.shape,  "default rate:", round(y_test.mean(), 4))

Train: (21000, 49) default rate: 0.2212
Test : (9000, 49) default rate: 0.2212


In [3]:
#Rows line up between raw and feature files
assert len(train) == len(X_train) and len(test) == len(X_test)
assert (train[target].values == y_train.values).all()
assert (test[target].values  == y_test.values).all()
assert list(X_train.columns) == list(X_test.columns)
print("Alignment OK")

Alignment OK


In [4]:
#Check whether identical customer profiles appear in both sets
raw_cols = [c for c in train.columns if c != target]
overlap = train.drop_duplicates(raw_cols).merge(test.drop_duplicates(raw_cols), on=raw_cols, how="inner")
print("Profiles present in both train and test:", len(overlap))

Profiles present in both train and test: 21


In [5]:
#Adversarial validation: Can a model distinguish whether a row came from the training dataset or the testing dataset?
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score

X_all = pd.concat([X_train, X_test], ignore_index=True)
is_test = np.r_[np.zeros(len(X_train)), np.ones(len(X_test))]

clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, verbose=-1, random_state=42)
p = cross_val_predict(clf, X_all, is_test,
                      cv=StratifiedKFold(5, shuffle=True, random_state=42),
                      method="predict_proba")[:, 1]
print("Adversarial AUC:", round(roc_auc_score(is_test, p), 3))

Adversarial AUC: 0.499


In [6]:
#Lock the test set 
def file_md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

fingerprints = {p: file_md5(p) for p in ["../data/train.csv", "../data/test.csv",
                                         "../data/features_train.csv", "../data/features_test.csv"]}
pd.Series(fingerprints).to_csv("../reports/data_fingerprints.csv", header=["md5"])
fingerprints

{'../data/train.csv': '69378212af39cdb238c018365507ed79',
 '../data/test.csv': '8c455118cbe92283033384900d373ff0',
 '../data/features_train.csv': '5ede39ac2586e5ae02a8ab9f26cba4db',
 '../data/features_test.csv': '0d28a3fe0bf541f0e0c7875b1ce2ce1b'}

#**Test-set protection:** The test set is locked at Step 5. It will be used only once for final model evaluation in Step 8. Checksums of the original and engineered datasets have been recorded.

In [7]:
# TEST SET IS LOCKED UNTIL STEP 8

In [8]:
#import functions from metrics.py
import importlib, metrics; importlib.reload(metrics)
from metrics import *

#Test the functions
rng = np.random.default_rng(0)
y_t = rng.integers(0, 2, 5000) #creates 5,000 artificial customers

# 1. Perfect scores: AUC=1, Gini=1, KS=1
print("Perfect:", auc(y_t, y_t), gini(y_t, y_t), ks_stat(y_t, y_t))

# 2. Random scores: AUC ~0.5, Gini ~0, KS small
r = rng.random(5000)
print("Random :", round(auc(y_t, r), 3), round(gini(y_t, r), 3), round(ks_stat(y_t, r), 3))

# 3. Reversed scores: AUC=0
print("Reversed AUC:", auc(y_t, 1 - y_t))

# 4. PSI of a sample against itself is 0, and a shifted sample is clearly > 0.25
x = rng.normal(0, 1, 20000)
print("PSI same:", round(psi(x, x), 4), "| PSI shifted by 1 sd:", round(psi(x, x + 1), 3))

# 5. Gini = 2*AUC - 1 holds
print(np.isclose(gini(y_t, r), 2 * auc(y_t, r) - 1))


Perfect: 1.0 1.0 1.0
Random : 0.512 0.024 0.029
Reversed AUC: 0.0
PSI same: 0.0 | PSI shifted by 1 sd: 0.94
True


In [9]:
#features with the largest PSI values.
psi_tbl = pd.Series({c: psi(X_train[c], X_test[c]) for c in X_train.columns}).sort_values(ascending=False)
psi_tbl.head(10)

util_avg                  0.002045
bill_avg_to_limit         0.002027
total_paid_over_billed    0.001870
pay_ratio_last3           0.001796
pay_amt_latest_vs_avg     0.001255
age                       0.001173
pay_amt_avg               0.001117
bill_cv                   0.001047
util_avg_last3            0.001030
bill_trend                0.000980
dtype: float64

In [10]:
#Choose the validation scheme
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Validation strategy

Model comparison and tuning will use 5-fold stratified cross-validation on the training set. Any preprocessing fitted on data, including imputation, scaling and WOE binning, must be fitted inside each training fold to avoid information leakage.

The dataset does not provide suitable dates for out-of-time validation. Cross-validation is therefore used as the development validation method, with this limitation documented.

In [11]:
#Baseline Models


In [12]:
#Import the required libraries
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_predict, cross_val_score

In [13]:
#Create a reusable function for cross-validation predictions
def cv_scores(model, cols, name):
    p = cross_val_predict(
        model,
        X_train[cols],
        y_train,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    return summarize(y_train, p, name), p

In [14]:
#Create a logistic regression function
def logit(C=1.0):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=C, max_iter=1000)
    )

In [15]:
#Baseline 0, “know nothing”:
b0, _ = cv_scores(DummyClassifier(strategy="prior"), ["delay_latest"], "B0: constant (prior)")

In [17]:
#Baseline 1, “one rule”: a single feature
b1, _ = cv_scores(logit(), ["delay_latest"], "B1: latest delay only")

In [18]:
#Baseline 2, “a credit analyst’s checklist”: a handful of obvious features, plain logistic regression
simple_cols = ["delay_latest", "n_late_months", "util_latest", "pay_ratio_latest",
               "n_paid_zero_with_bill", "log_limit"]
b2, p_b2 = cv_scores(logit(), simple_cols, "B2: 6-feature logistic")

In [19]:
#Baseline 3, “everything, no cleverness”: all features in a regularized logistic regression
all_cols = list(X_train.columns)
b3, _ = cv_scores(logit(C=0.1), all_cols, "B3: all features, logistic (L2)")

In [20]:
#Baseline 4, “unconstrained ML”: a quick LightGBM at default-ish settings
import lightgbm as lgb
b4, _ = cv_scores(lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
                                     verbose=-1, random_state=42), all_cols, "B4: LightGBM (quick)")

In [21]:
#Put them in one table
baselines = pd.DataFrame([b0, b1, b2, b3, b4]).set_index("model").round(3)
baselines

,AUC,Gini,KS,Brier
model,,,,
B0: constant (prior),0.500,0.000,0.000,0.172
B1: latest delay only,0.699,0.397,0.377,0.142
B2: 6-feature logistic,0.759,0.517,0.415,0.138
"B3: all features, logistic (L2)",0.782,0.565,0.429,0.134
B4: LightGBM (quick),0.785,0.571,0.432,0.133


In [22]:
#Fold stability check
from sklearn.model_selection import cross_val_score
folds = cross_val_score(logit(), X_train[simple_cols], y_train, cv=cv, scoring="roc_auc")
print("B2 AUC per fold:", folds.round(4), "| std:", folds.std().round(4))

B2 AUC per fold: [0.7709 0.772  0.7481 0.7464 0.7574] | std: 0.0109


In [23]:
#Plot a decile table for B2 to see the business shape
dt = decile_table(y_train, p_b2)
dt.round(3)

,customers,bads,avg_pd,bad_rate,cum_bads_captured_%
decile,,,,,
10,2100,1451,0.699,0.691,31.238
9,2100,887,0.408,0.422,50.334
8,2100,550,0.233,0.262,62.174
7,2100,391,0.167,0.186,70.592
6,2100,318,0.145,0.151,77.438
5,2100,220,0.131,0.105,82.174
4,2100,238,0.119,0.113,87.298
3,2100,199,0.111,0.095,91.582
2,2100,214,0.105,0.102,96.189


In [24]:
#Results log and commit
results_log = baselines.reset_index().assign(stage="baseline (5-fold CV, train)")
results_log.to_csv("../reports/model_results_log.csv", index=False)